# CI-ready machine-learning pipeline

A small, deterministic classification example used to test notebook execution and output persistence in GitHub Actions. It uses scikit-learn’s built-in Iris dataset, so no external dataset download is required.

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path

import pandas as pd
from sklearn.datasets import load_iris
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
dataset = load_iris(as_frame=True)
X = dataset.data
y = dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)
predictions = model.predict(X_test)
accuracy = float(accuracy_score(y_test, predictions))
print(f"Dataset: Iris | train={len(X_train)} | test={len(X_test)}")
print(f"Test accuracy: {accuracy:.4f}")
print(classification_report(y_test, predictions, target_names=dataset.target_names))


In [ ]:
importance = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42, scoring="accuracy"
)
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False)
importance_df.to_csv(OUTPUT_DIR / "feature_importance.csv", index=False)
summary = {
    "status": "success",
    "dataset": "Iris",
    "model": "StandardScaler + LogisticRegression",
    "test_accuracy": accuracy,
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "features": list(X.columns),
    "run_utc": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_DIR / "ml_summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
display(importance_df)
print("Saved outputs/ml_summary.json and outputs/feature_importance.csv")
